# The Purpose of this file is to verify whether the direct auto diff on orientation is feasible

In [80]:
import jax
import jax.numpy as jnp
from jax import grad

# JAX AUTO DIFF

In [81]:
def f(x):
    return x**2 + 3*x + 2

grad_f = jax.grad(f) 
print(grad_f(2.0)) 

7.0


# JAX Lie

## 1 Manual Lie

In [82]:
from jax.scipy.spatial.transform import Rotation as R
def loss_fn(q): 
    """loss function is the gravity vector in the body frame

    Args:
        q (_type_): _description_

    Returns:
        _type_: _description_
    """
    p=jnp.array([0.0,0.0,1]) 
    q = jnp.roll(q, -1) # wxyz -> xyzw
    r = R.from_quat(q) 
    rot_matrix = r.as_matrix()
    # print(rot_matrix)
    new_p=rot_matrix @ p

    area=jnp.dot(new_p,p)
    return area


q = jnp.array([0.766,0.00,0.643,0.00])  # #ZYX order, pitch 80 degree
area=loss_fn(q)
# print(area)

grad_loss_fn = grad(loss_fn)
# correspond lie algebra
omega = jnp.array([0.0, 0.523, 0.0])  # so(3) (wx, wy, wz)
grad_q = grad_loss_fn(q)

# # quaternion gradient to the lie algebra gradient
grad_omega = 2 * grad_q[1:]  # 只取后三个分量
print("四元数梯度:", grad_q)
print("李代数梯度:", grad_omega)

四元数梯度: [ 1.2662888  0.        -1.5085181  0.       ]
李代数梯度: [ 0.        -3.0170362  0.       ]


## 2   JAXLIE lib

In [83]:
from jaxlie import SO3
from jax import grad
import numpy as np
q=SO3.from_quaternion_xyzw(jnp.array([0.00,0.643,0.00, 0.766]))  # #ZYX order, pitch 30 degree
print("lie_group",q)

omega = SO3.log(q)
print("lie_algebra",omega)

def loss_fn_jax_l(q):
    p=jnp.array([0.0,0.0,1]) 
    rot_matrix=SO3.as_matrix(q)
    new_p=rot_matrix @ p
    area=jnp.dot(new_p,p)
    return area

area=loss_fn_jax_l(q)
# print(area)
grad_loss_fn_jax_l = grad(loss_fn_jax_l)
grad_q = grad_loss_fn_jax_l(q)
print(grad_q)

lie_group SO3(wxyz=[0.766 0.    0.643 0.   ])
lie_algebra [0.        1.3966459 0.       ]
SO3(wxyz=[ 1.26629  0.      -1.50852  0.     ])


## seems like two methods has no difference, but why it is necessary to do differentiation on the SLAM?